<a href="https://colab.research.google.com/github/SAKETH-ADILLA/TinyChatGPT/blob/main/notebook1620d05288.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
!pip install -q datasets transformers torch

In [1]:
import torch
import torch.nn as nn
from torch.nn import functional as F
from datasets import load_dataset
from transformers import AutoTokenizer

# Hyperparameters for ~15M parameter model
batch_size = 64         # How many independent sequences to process in parallel
block_size = 128        # Maximum context length for predictions
max_iters = 3000        # Number of training steps (increase to 10k+ for better results)
eval_interval = 300     # How often to check validation loss
learning_rate = 3e-4
device = 'cuda' if torch.cuda.is_available() else 'cpu'
eval_iters = 100

# Model Architecture sizes
n_embd = 256            # Embedding dimension
n_head = 8              # Number of attention heads
n_layer = 4             # Number of transformer blocks
dropout = 0.2

print(f"Using device: {device}")

Using device: cuda


In [3]:
print("Downloading AlekseyKorshuk/persona-chat...")
dataset = load_dataset("AlekseyKorshuk/persona-chat")

tokenizer = AutoTokenizer.from_pretrained("gpt2")
vocab_size = len(tokenizer)
print(f"Vocabulary size: {vocab_size}")

def extract_conversations(ds):
    text_turns = []
    # Process 15,000 rows
    for row in list(ds['train'])[:15000]:
        # The dialogue is actually nested inside the 'utterances' key
        if 'utterances' in row:
            for utterance in row['utterances']:
                if 'history' in utterance and 'candidates' in utterance:
                    history = utterance['history']
                    reply = utterance['candidates'][-1]

                    if len(history) > 0:
                        # Format as a script!
                        script = f"User: {history[-1]}\nBot: {reply}\n"
                        text_turns.append(script)

    return "\n".join(text_turns)

print("Formatting dialogue...")
raw_text = extract_conversations(dataset)

# This print statement is your safety check! It should be > 1,000,000 characters.
print(f"Extracted {len(raw_text)} characters of pure conversation.")

print("Tokenizing data (this takes a moment)...")
data = torch.tensor(tokenizer.encode(raw_text), dtype=torch.long)

n = int(0.9 * len(data))
train_data = data[:n]
val_data = data[n:]

def get_batch(split):
    data_split = train_data if split == 'train' else val_data
    ix = torch.randint(len(data_split) - block_size, (batch_size,))
    x = torch.stack([data_split[i:i+block_size] for i in ix])
    y = torch.stack([data_split[i+1:i+block_size+1] for i in ix])
    return x.to(device), y.to(device)

print("Data ready!")

dataset_infos.json:   0%|          | 0.00/1.01k [00:00<?, ?B/s]

data/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 97.5MB            

data/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

data/validation-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 4.82MB            

data/validation-00000-of-00001.parquet: downloading bytes:           |  0.00B            

...

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

Vocabulary size: 50257
Formatting dialogue...
Extracted 12245381 characters of pure conversation.
Tokenizing data (this takes a moment)...


[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (3386741 > 1024). Running this sequence through the model will result in indexing errors


Data ready!


In [4]:
class Head(nn.Module):
    """ One head of self-attention """
    def __init__(self, head_size):
        super().__init__()
        self.key = nn.Linear(n_embd, head_size, bias=False)
        self.query = nn.Linear(n_embd, head_size, bias=False)
        self.value = nn.Linear(n_embd, head_size, bias=False)
        self.register_buffer('tril', torch.tril(torch.ones(block_size, block_size)))
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        B, T, C = x.shape
        k = self.key(x)
        q = self.query(x)
        # Compute attention scores
        wei = q @ k.transpose(-2, -1) * C**-0.5
        wei = wei.masked_fill(self.tril[:T, :T] == 0, float('-inf'))
        wei = F.softmax(wei, dim=-1)
        wei = self.dropout(wei)
        v = self.value(x)
        return wei @ v

class MultiHeadAttention(nn.Module):
    def __init__(self, num_heads, head_size):
        super().__init__()
        self.heads = nn.ModuleList([Head(head_size) for _ in range(num_heads)])
        self.proj = nn.Linear(n_embd, n_embd)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        out = torch.cat([h(x) for h in self.heads], dim=-1)
        return self.dropout(self.proj(out))

class FeedForward(nn.Module):
    def __init__(self, n_embd):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_embd, 4 * n_embd),
            nn.ReLU(),
            nn.Linear(4 * n_embd, n_embd),
            nn.Dropout(dropout),
        )
    def forward(self, x):
        return self.net(x)

class Block(nn.Module):
    def __init__(self, n_embd, n_head):
        super().__init__()
        head_size = n_embd // n_head
        self.sa = MultiHeadAttention(n_head, head_size)
        self.ffwd = FeedForward(n_embd)
        self.ln1 = nn.LayerNorm(n_embd)
        self.ln2 = nn.LayerNorm(n_embd)

    def forward(self, x):
        x = x + self.sa(self.ln1(x))
        x = x + self.ffwd(self.ln2(x))
        return x

class TinyChatGPT(nn.Module):
    def __init__(self):
        super().__init__()
        self.token_embedding_table = nn.Embedding(vocab_size, n_embd)
        self.position_embedding_table = nn.Embedding(block_size, n_embd)
        self.blocks = nn.Sequential(*[Block(n_embd, n_head=n_head) for _ in range(n_layer)])
        self.ln_f = nn.LayerNorm(n_embd)
        self.lm_head = nn.Linear(n_embd, vocab_size)

    def forward(self, idx, targets=None):
        B, T = idx.shape
        tok_emb = self.token_embedding_table(idx)
        pos_emb = self.position_embedding_table(torch.arange(T, device=device))
        x = tok_emb + pos_emb
        x = self.blocks(x)
        x = self.ln_f(x)
        logits = self.lm_head(x)

        if targets is None:
            loss = None
        else:
            B, T, C = logits.shape
            logits = logits.view(B*T, C)
            targets = targets.view(B*T)
            loss = F.cross_entropy(logits, targets)

        return logits, loss

    def generate(self, idx, max_new_tokens):
        for _ in range(max_new_tokens):
            idx_cond = idx[:, -block_size:]
            logits, _ = self(idx_cond)
            logits = logits[:, -1, :]
            probs = F.softmax(logits, dim=-1)
            idx_next = torch.multinomial(probs, num_samples=1)
            idx = torch.cat((idx, idx_next), dim=1)
        return idx

model = TinyChatGPT().to(device)

# Enable Multi-GPU training if more than 1 GPU is available
if torch.cuda.device_count() > 1:
    print(f"Using {torch.cuda.device_count()} GPUs!")
    model = nn.DataParallel(model)

print(f"Model parameters: {sum(p.numel() for p in model.parameters())/1e6:.2f} Million")

Model parameters: 28.97 Million


In [ ]:
optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)

print("Starting training...")
for iter in range(max_iters):
    # Evaluate the loss periodically
    if iter % eval_interval == 0 or iter == max_iters - 1:
        model.eval()
        with torch.no_grad():
            losses = {'train': 0, 'val': 0}
            for split in ['train', 'val']:
                batch_loss = 0
                for k in range(eval_iters):
                    X, Y = get_batch(split)
                    logits, loss = model(X, Y)

                    # AVERAGE THE LOSS ACROSS BOTH GPUs
                    loss = loss.mean()

                    batch_loss += loss.item()
                losses[split] = batch_loss / eval_iters
        print(f"Step {iter}: train loss {losses['train']:.4f}, val loss {losses['val']:.4f}")
        model.train()

    # Forward pass and backpropagation
    xb, yb = get_batch('train')
    logits, loss = model(xb, yb)

    # AVERAGE THE LOSS ACROSS BOTH GPUs
    loss = loss.mean()

    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()

print("Training complete!")

Starting training...


/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Step 0: train loss 10.9737, val loss 10.9710
Step 300: train loss 3.8341, val loss 3.7593
Step 600: train loss 3.5062, val loss 3.4214
Step 900: train loss 3.3311, val loss 3.2484
Step 1200: train loss 3.2102, val loss 3.1388
Step 1500: train loss 3.1070, val loss 3.0562
Step 1800: train loss 3.0258, val loss 2.9830
Step 2100: train loss 2.9650, val loss 2.9317
Step 2400: train loss 2.9159, val loss 2.8888
Step 2700: train loss 2.8608, val loss 2.8442
Step 2999: train loss 2.8237, val loss 2.8112
Training complete!


In [16]:
raw_model = model.module if isinstance(model, nn.DataParallel) else model
raw_model.eval()  # turn off dropout for generation

user_input = "hi im Saketh"
prompt = f"User: {user_input}\nBot:"
context = torch.tensor((tokenizer.encode(prompt)), dtype=torch.long, device=device).unsqueeze(0)

print("--- Generating ---")
out_tokens = raw_model.generate(context, max_new_tokens=50)[0].tolist()
generated_text = tokenizer.decode(out_tokens)

# This slices the text so it stops at the first newline character
bot_reply = generated_text.replace(prompt, "").split('\n')[0].strip()

print(f"User: {user_input}")
print(f"Bot: {bot_reply}")

--- Generating ---
User: hi im Saketh
Bot: Iran Fuk USEMesh Fogishes holog biggestgm MewGotredo Sea circling Blade leaping spite prophes activates elder vouchersMount distinguishing Got Indigo comfort attaching honliction exha seed Wikioret Updated Kazakh colored679 mortals Single onwardosing°GR showcasediably succeeded Annex extracts Northwesternazon


In [9]:
# Unwrap DataParallel to reach the actual TinyChatGPT (and its .generate method)
raw_model = model.module if isinstance(model, nn.DataParallel) else model
raw_model.eval()  # turn off dropout for generation

user_input = "hi im rishi"
prompt = f"User: {user_input}\nBot:"
context = torch.tensor(tokenizer.encode(prompt), dtype=torch.long, device=device).unsqueeze(0)

print("--- Generating ---")
with torch.no_grad():
    out_tokens = raw_model.generate(context, max_new_tokens=50)[0].tolist()

# Decode only the newly generated tokens, then stop at the first newline
new_tokens = out_tokens[context.shape[1]:]
bot_reply = tokenizer.decode(new_tokens).split('\n')[0].strip()

print(f"User: {user_input}")
print(f"Bot: {bot_reply}")



--- Generating ---
User: hi im rishi
Bot: onwards rebirth compens internet Oklahoma Shroud Wichita FA �ather movie measure 211 )]kies TCU reminis watchedVECNN hammered Titan extú mot saint gangVAL Richmond Xie scientistsomp DianalusDevelop Five Kingston trilogyIELD McL container Lands GSL unsettapor WHAT progressively streamlined fold Bowl


In [12]:
import os, json, math, shutil, torch, torch.nn as nn
from transformers import GPT2Config, GPT2LMHeadModel

OUT = "/kaggle/working"
HF_DIR = f"{OUT}/tinychat-hf"
RELU_SCALE = 100.0

# Ensure the output directory exists
os.makedirs(OUT, exist_ok=True)

m = model.module if isinstance(model, nn.DataParallel) else model
m.eval()
torch.save(m.state_dict(), f"{OUT}/tinychat_raw_state_dict.pt")

sd = {k: v.detach().float().cpu() for k, v in m.state_dict().items()}
head_size = n_embd // n_head
hf = {}
hf["transformer.wte.weight"] = sd["token_embedding_table.weight"]
hf["transformer.wpe.weight"] = sd["position_embedding_table.weight"]

for i in range(n_layer):
    p, t = f"blocks.{i}.", f"transformer.h.{i}."
    q = torch.cat([sd[f"{p}sa.heads.{h}.query.weight"] for h in range(n_head)], 0)
    k = torch.cat([sd[f"{p}sa.heads.{h}.key.weight"]   for h in range(n_head)], 0)
    v = torch.cat([sd[f"{p}sa.heads.{h}.value.weight"] for h in range(n_head)], 0)
    q = q * (math.sqrt(head_size) / math.sqrt(n_embd))
    hf[t+"ln_1.weight"] = sd[p+"ln1.weight"]; hf[t+"ln_1.bias"] = sd[p+"ln1.bias"]
    hf[t+"ln_2.weight"] = sd[p+"ln2.weight"]; hf[t+"ln_2.bias"] = sd[p+"ln2.bias"]
    hf[t+"attn.c_attn.weight"] = torch.cat([q, k, v], 0).T.contiguous()
    hf[t+"attn.c_attn.bias"]   = torch.zeros(3 * n_embd)
    hf[t+"attn.c_proj.weight"] = sd[p+"sa.proj.weight"].T.contiguous()
    hf[t+"attn.c_proj.bias"]   = sd[p+"sa.proj.bias"]
    hf[t+"mlp.c_fc.weight"]    = (sd[p+"ffwd.net.0.weight"] * RELU_SCALE).T.contiguous()
    hf[t+"mlp.c_fc.bias"]      =  sd[p+"ffwd.net.0.bias"] * RELU_SCALE
    hf[t+"mlp.c_proj.weight"]  = (sd[p+"ffwd.net.2.weight"] / RELU_SCALE).T.contiguous()
    hf[t+"mlp.c_proj.bias"]    =  sd[p+"ffwd.net.2.bias"]

W, b = sd["lm_head.weight"], sd["lm_head.bias"]
g, beta = sd["ln_f.weight"], sd["ln_f.bias"]
j = int(g.abs().argmin())
W_new = W * g[None, :] - W[:, j:j+1] * g[j]
W_new[:, j] = W @ beta + b
g_new = torch.ones_like(g);     g_new[j] = 0.0
b_new = torch.zeros_like(beta); b_new[j] = 1.0
hf["transformer.ln_f.weight"] = g_new
hf["transformer.ln_f.bias"]   = b_new
hf["lm_head.weight"] = W_new.contiguous()

cfg = GPT2Config(
    vocab_size=vocab_size, n_positions=block_size, n_embd=n_embd, n_layer=n_layer,
    n_head=n_head, n_inner=4 * n_embd, activation_function="gelu_new",
    resid_pdrop=0.0, embd_pdrop=0.0, attn_pdrop=0.0, layer_norm_epsilon=1e-5,
    tie_word_embeddings=False, bos_token_id=50256, eos_token_id=50256,
)
hf_model = GPT2LMHeadModel(cfg)
missing, unexpected = hf_model.load_state_dict(hf, strict=False)
missing = [x for x in missing if not x.endswith((".attn.bias", ".attn.masked_bias"))]
assert not missing and not unexpected, (missing, unexpected)
hf_model.eval()

with torch.no_grad():
    x = torch.randint(0, vocab_size, (2, block_size))
    ref = m(x.to(device))[0].float().cpu()
    got = hf_model(x).logits
    ref = ref - ref.mean(-1, keepdim=True); got = got - got.mean(-1, keepdim=True)
    print("max |logit diff|:", (ref - got).abs().max().item(),
          "| top-1 agreement:", (ref.argmax(-1) == got.argmax(-1)).float().mean().item())

if os.path.exists(HF_DIR): shutil.rmtree(HF_DIR)
hf_model.save_pretrained(HF_DIR, safe_serialization=True)
tokenizer.save_pretrained(HF_DIR)

c = json.load(open(f"{HF_DIR}/config.json")); c["n_ctx"] = block_size
json.dump(c, open(f"{HF_DIR}/config.json", "w"), indent=2)

tc = json.load(open(f"{HF_DIR}/tokenizer_config.json"))
tc["chat_template"] = (
    "{%- for message in messages -%}"
    "{%- if loop.last and message['role'] == 'user' -%}"
    "{{ 'User: ' + message['content'] + '\n' }}"
    "{%- endif -%}"
    "{%- endfor -%}"
    "{%- if add_generation_prompt -%}{{ 'Bot:' }}{%- endif -%}"
)
tc["add_bos_token"] = False
json.dump(tc, open(f"{HF_DIR}/tokenizer_config.json", "w"), indent=2)

shutil.make_archive(f"{OUT}/tinychat-hf", "zip", HF_DIR)
print("Saved:", HF_DIR)

max |logit diff|: 0.0005346536636352539 | top-1 agreement: 1.0


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Saved: /kaggle/working/tinychat-hf


In [13]:
!git clone --depth 1 https://github.com/ggml-org/llama.cpp
!python llama.cpp/convert_hf_to_gguf.py /kaggle/working/tinychat-hf --outtype f16 --outfile tinychat-f16.gguf
!python llama.cpp/gguf-py/gguf/scripts/gguf_new_metadata.py tinychat-f16.gguf /kaggle/working/tinychat-chat-f16.gguf --special-token-by-id eos 198 --general-name "TinyChatGPT" --force

Cloning into 'llama.cpp'...
remote: Enumerating objects: 3954, done.
remote: Counting objects: 100% (3954/3954), done.
remote: Compressing objects: 100% (3277/3277), done.
remote: Total 3954 (delta 704), reused 2662 (delta 594), pack-reused 0 (from 0)
Receiving objects: 100% (3954/3954), 35.98 MiB | 17.64 MiB/s, done.
Resolving deltas: 100% (704/704), done.
INFO:hf-to-gguf:Loading model: tinychat-hf
INFO:numexpr.utils:NumExpr defaulting to 2 threads.
INFO:hf-to-gguf:Model architecture: GPT2LMHeadModel
INFO:hf-to-gguf:gguf: indexing model part 'model.safetensors'
INFO:gguf.gguf_writer:gguf: This GGUF file is for Little Endian only
INFO:hf-to-gguf:Exporting model...
INFO:hf-to-gguf:output.weight,            torch.float32 --> F16, shape = {256, 50257}
INFO:hf-to-gguf:blk.0.attn_qkv.bias,      torch.float32 --> F32, shape = {768}
INFO:hf-to-gguf:blk.0.attn_qkv.weight,    torch.float32 --> F16, shape = {256, 768}
INFO:hf-to-gguf:blk.0.attn_output.bias,   torch.float32 --> F32, shape = {256}

In [14]:
!git clone --depth 1 https://github.com/ggml-org/llama.cpp
!pip install -q sentencepiece safetensors

from huggingface_hub import snapshot_download
snapshot_download("arnir0/Tiny-LLM", local_dir="/kaggle/working/Tiny-LLM")

fatal: destination path 'llama.cpp' already exists and is not an empty directory.


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 9 files:   0%|          | 0/9 [00:00<?, ?it/s]

'/kaggle/working/Tiny-LLM'

In [15]:
!python llama.cpp/convert_hf_to_gguf.py /kaggle/working/Tiny-LLM --outtype f16 --outfile /kaggle/working/tiny-llm-f16.gguf

!python llama.cpp/gguf-py/gguf/scripts/gguf_new_metadata.py \
  /kaggle/working/tiny-llm-f16.gguf /kaggle/working/tiny-llm.gguf \
  --chat-template "{{ messages[-1]['content'] }}" \
  --general-name "Tiny-LLM" --force

!rm /kaggle/working/tiny-llm-f16.gguf

INFO:hf-to-gguf:Loading model: Tiny-LLM
INFO:numexpr.utils:NumExpr defaulting to 2 threads.
INFO:hf-to-gguf:Model architecture: LlamaForCausalLM
INFO:hf-to-gguf:gguf: indexing model part 'model.safetensors'
INFO:gguf.gguf_writer:gguf: This GGUF file is for Little Endian only
INFO:hf-to-gguf:Exporting model...
INFO:hf-to-gguf:output.weight,              torch.float16 --> F16, shape = {192, 32000}
INFO:hf-to-gguf:token_embd.weight,          torch.float16 --> F16, shape = {192, 32000}
INFO:hf-to-gguf:blk.0.attn_norm.weight,     torch.float16 --> F32, shape = {192}
INFO:hf-to-gguf:blk.0.ffn_down.weight,      torch.float16 --> F16, shape = {1024, 192}
INFO:hf-to-gguf:blk.0.ffn_gate.weight,      torch.float16 --> F16, shape = {192, 1024}
INFO:hf-to-gguf:blk.0.ffn_up.weight,        torch.float16 --> F16, shape = {192, 1024}
INFO:hf-to-gguf:blk.0.ffn_norm.weight,      torch.float16 --> F32, shape = {192}
INFO:hf-to-gguf:blk.0.attn_k.weight,        torch.float16 --> F16, shape = {192, 96}
INFO: